we implement the Mamdani fuzzy inference system from scratch without any fuzzy libraries

the pipeline follows three main steps: fuzzification, inference, and defuzzification

input variables: nkill, nwound, propextent, attack_encoded, weapon_encoded
output variable: severity_index (0 to 100)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def trimf(x, a, b, c):
    return np.maximum(0, np.minimum((x - a) / (b - a + 1e-9),
                                     (c - x) / (c - b + 1e-9)))

def trapmf(x, a, b, c, d):
    return np.maximum(0, np.minimum(
        np.minimum((x - a) / (b - a + 1e-9), 1),
        (d - x) / (d - c + 1e-9)
    ))

def fuzzify_nkill(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 0, 0, 1, 4)[0]),
        "Medium":  float(trimf(x, 2, 6, 12)[0]),
        "High":    float(trimf(x, 6, 15, 30)[0]),
        "Extreme": float(trapmf(x, 25, 40, 50, 50)[0]),
    }

def fuzzify_nwound(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 0, 0, 2, 6)[0]),
        "Medium":  float(trimf(x, 3, 10, 20)[0]),
        "High":    float(trimf(x, 15, 35, 60)[0]),
        "Extreme": float(trapmf(x, 45, 65, 80, 80)[0]),
    }

def fuzzify_propextent(val):
    x = np.array([val], dtype=float)
    return {
        "None":         float(trapmf(x, 0, 0, 0, 0.5)[0]),
        "Minor":        float(trimf(x, 0.5, 1, 1.5)[0]),
        "Major":        float(trimf(x, 1.5, 2, 2.5)[0]),
        "Catastrophic": float(trapmf(x, 2.5, 2.8, 4, 4)[0]),
    }

def fuzzify_attack(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 1, 1, 1, 1.8)[0]),
        "Medium":  float(trimf(x, 1.5, 2, 2.5)[0]),
        "High":    float(trimf(x, 2.5, 3, 3.5)[0]),
        "Extreme": float(trapmf(x, 2.8, 3.5, 5, 5)[0]),
    }

def fuzzify_weapon(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 1, 1, 1, 1.8)[0]),
        "Medium":  float(trimf(x, 1.5, 2, 2.5)[0]),
        "High":    float(trimf(x, 2.5, 3, 3.5)[0]),
        "Extreme": float(trapmf(x, 2.8, 3.5, 5, 5)[0]),
    }

In [ ]:
def assign_severity(k, w, p, a, wp):
    kill_score  = {"Low": 0, "Medium": 1, "High": 2, "Extreme": 3}
    wound_score = {"Low": 0, "Medium": 1, "High": 2, "Extreme": 3}
    prop_score  = {"None": 0, "Minor": 1, "Major": 2, "Catastrophic": 3}
    atk_score   = {"Low": 0, "Medium": 1, "High": 2, "Extreme": 3}
    wpn_score   = {"Low": 0, "Medium": 1, "High": 2, "Extreme": 3}

    score = (
        0.35 * kill_score[k]  +
        0.25 * wound_score[w] +
        0.15 * prop_score[p]  +
        0.15 * atk_score[a]   +
        0.10 * wpn_score[wp]
    )

    if score < 0.75:
        return "Low"
    elif score < 1.5:
        return "Medium"
    elif score < 2.25:
        return "High"
    else:
        return "Critical"

kill_levels = ["Low", "Medium", "High", "Extreme"]
prop_levels = ["None", "Minor", "Major", "Catastrophic"]

rules = []
for k in kill_levels:
    for w in kill_levels:
        for p in prop_levels:
            for a in kill_levels:
                for wp in kill_levels:
                    sev = assign_severity(k, w, p, a, wp)
                    rules.append((k, w, p, a, wp, sev))

print(f"Total rules: {len(rules)}")

## output universe & MF

In [ ]:
x_out = np.linspace(0, 100, 1000)

output_mf = {
    "Low":      trapmf(x_out, 0, 0, 15, 30),
    "Medium":   trimf(x_out, 20, 40, 55),
    "High":     trimf(x_out, 45, 60, 75),
    "Critical": trapmf(x_out, 65, 80, 100, 100),
}

## mamdani inference function

In [ ]:
def mamdani_infer(nkill_val, nwound_val, prop_val, atk_val, wpn_val):
    fk = fuzzify_nkill(nkill_val)
    fw = fuzzify_nwound(nwound_val)
    fp = fuzzify_propextent(prop_val)
    fa = fuzzify_attack(atk_val)
    fwp = fuzzify_weapon(wpn_val)

    aggregated = np.zeros_like(x_out)

    for (k, w, p, a, wp, out) in rules:
        strength = min(fk[k], fw[w], fp[p], fa[a], fwp[wp])
        clipped = np.minimum(strength, output_mf[out])
        aggregated = np.maximum(aggregated, clipped)

    return aggregated

def defuzzify_centroid(aggregated):
    denom = np.sum(aggregated)
    if denom == 0:
        return 0.0
    return float(np.sum(x_out * aggregated) / denom)

## single inference example

testing the system on one sample input before running on the full dataset

In [ ]:
# test single input

# High kill, Medium wound, Major damage, High attack, Extreme weapon
nkill_test  = 20   # High
nwound_test = 10   # Medium
prop_test   = 2    # Major (prop_inverted)
atk_test    = 3    # High
wpn_test    = 4    # Extreme

agg = mamdani_infer(nkill_test, nwound_test, prop_test, atk_test, wpn_test)
result = defuzzify_centroid(agg)

print(f"Input  : nkill={nkill_test}, nwound={nwound_test}, propextent={prop_test}, attack={atk_test}, weapon={wpn_test}")
print(f"Output : severity_score = {result:.2f}")

if result < 25:
    label = "Low"
elif result < 50:
    label = "Medium"
elif result < 75:
    label = "High"
else:
    label = "Critical"

print(f"Label  : {label}")

plt.figure(figsize=(10, 4))
plt.plot(x_out, agg, color="#c0392b", linewidth=2, label="Aggregated output")
plt.axvline(result, color="black", linestyle="--", label=f"Centroid = {result:.2f}")
plt.fill_between(x_out, agg, alpha=0.2, color="#c0392b")
plt.title("Mamdani Defuzzification: Aggregated Output")
plt.xlabel("Severity score")
plt.ylabel("Membership degree")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv("../data/gtd_processed.csv")

prop_map = {1: 3, 2: 2, 3: 1, 4: 0}
df["prop_inverted"] = df["propextent"].map(prop_map)

print(f"Loaded {len(df):,} rows")
df[["nkill", "nwound", "propextent", "attack_encoded",
    "weapon_encoded", "severity_index"]].head()

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

def mamdani_predict(row):
    agg = mamdani_infer(
        row["nkill"], row["nwound"], row["prop_inverted"],
        row["attack_encoded"], row["weapon_encoded"]
    )
    return defuzzify_centroid(agg)

df["mamdani_score"] = df.progress_apply(mamdani_predict, axis=1)

def score_to_label(score):
    if score < 25:
        return "Low"
    elif score < 50:
        return "Medium"
    elif score < 75:
        return "High"
    else:
        return "Critical"

df["mamdani_label"] = df["mamdani_score"].apply(score_to_label)
print(df["mamdani_label"].value_counts())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_true = df["severity_index"]
y_pred = df["mamdani_label"]
order  = ["Low", "Medium", "High", "Critical"]

acc = accuracy_score(y_true, y_pred)
print(f"Mamdani Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(y_true, y_pred, labels=order, target_names=order))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

order = ["Low", "Medium", "High", "Critical"]
cm = confusion_matrix(y_true, y_pred, labels=order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=order)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Reds", colorbar=False)
plt.title("Mamdani Confusion Matrix")
plt.tight_layout()
plt.show()